# Sprint 7 - Kalman and OU

This notebook demonstrates dynamic hedge-ratio estimation and OU diagnostics with deterministic synthetic data. It is a research workflow example, not a market backtest and not live-trading permission.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import math
import numpy as np
import pandas as pd

from src.research import (
    AnalysisScope,
    KalmanFilterConfig,
    assess_stationarity,
    estimate_ou,
    fit_kalman_filter,
    rolling_zscore,
)

## Synthetic Pair

The dependent leg is generated as `y_t = beta_t * x_t + alpha_t + spread_t`, where the spread follows a simple mean-reverting process.

In [ ]:
periods = 360
rng_x = np.random.default_rng(7)
rng_spread = np.random.default_rng(17)
x = 100.0 + np.cumsum(rng_x.normal(0.0, 0.35, size=periods))
beta_true = 1.35 + np.sin(np.arange(periods) / 90.0) * 0.05
alpha_true = -4.0

spread = np.zeros(periods)
for index in range(1, periods):
    spread[index] = 0.82 * spread[index - 1] + rng_spread.normal(0.0, 0.18)

y = beta_true * x + alpha_true + spread
synthetic_pair = pd.DataFrame({"x": x, "y": y, "beta_true": beta_true, "spread_true": spread})
synthetic_pair.head()

## Kalman Dynamic Hedge Ratio

The implementation updates sequentially. The beta, alpha, innovation, and spread arrays have the same length as the input series.

In [ ]:
kalman = fit_kalman_filter(
    y,
    x,
    KalmanFilterConfig(
        observation_variance=0.05,
        beta_state_variance=1e-5,
        alpha_state_variance=1e-4,
        initial_beta=1.0,
        initial_alpha=0.0,
        unstable_beta_abs_threshold=5.0,
    ),
)

kalman_table = synthetic_pair.assign(
    beta_kalman=kalman.beta,
    alpha_kalman=kalman.alpha,
    spread_kalman=kalman.spread,
    innovation=kalman.innovation,
)

kalman_table.tail()

## Stationarity, OU, and Z-Score

Full-sample stationarity and OU estimates are exploratory diagnostics. The section below uses the known synthetic OU spread component so the statistical checks have a controlled target. Rolling z-score is shifted one row and is marked no-look-ahead by the module.

In [ ]:
spread_series = pd.Series(spread[40:], name="spread_true_ou_component")
stationarity = assess_stationarity(
    spread_series,
    max_half_life=80.0,
    min_observations=80,
    scope=AnalysisScope.FULL_SAMPLE_EXPLORATORY,
)
ou_fit = estimate_ou(spread_series, dt=1.0, min_observations=80)
zscore = rolling_zscore(spread_series, window=48)

summary = pd.DataFrame(
    [
        {
            "stationarity_status": stationarity.status.value,
            "adf_p_value": stationarity.adf.p_value,
            "kpss_p_value": stationarity.kpss.p_value,
            "preliminary_half_life": stationarity.half_life.half_life,
            "ou_status": ou_fit.status.value,
            "ou_theta": ou_fit.theta,
            "ou_mu": ou_fit.mu,
            "ou_sigma": ou_fit.sigma,
            "ou_half_life": ou_fit.half_life,
            "latest_rolling_zscore": zscore.dropna().iloc[-1],
            "zscore_lookahead_safe": zscore.attrs["lookahead_safe"],
            "kalman_beta_unstable": kalman.beta_unstable,
        }
    ]
)

summary

## Review Notes

- `assess_stationarity` and `estimate_ou` here are full-sample exploratory diagnostics.
- A real walk-forward experiment must refit or update parameters only with data available before each decision timestamp.
- This notebook does not implement backtest, paper trading, live trading, XGBoost, `P_fill`, or `P_profit_given_fill`.